In [73]:
import pandas as pd

df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

df_train.head()

,aline_flg,age,gender_num,weight_first,bmi,sapsi_first,sofa_first,service_unit,service_num,day_icu_intime,...,platelet_first,sodium_first,potassium_first,tco2_first,chloride_first,bun_first,creatinine_first,po2_first,pco2_first,iv_day_1
0,1,72.36841,1.0,75.0,29.912791,15,9,SICU,1,Friday,...,354.0,138.0,4.6,15.0,109.0,41.0,1.6,196.0,39.0,2230.875000
1,0,48.40044,0.0,NaN,NaN,14,5,MICU,0,Monday,...,367.0,133.0,3.7,19.0,98.0,25.0,0.9,473.0,32.0,2850.000000
2,1,48.97547,1.0,97.0,31.579563,13,11,SICU,1,Thursday,...,145.0,133.0,4.0,31.0,95.0,18.0,1.0,493.0,34.0,3682.880127
3,1,60.37829,0.0,51.0,25.207321,18,7,SICU,1,Saturday,...,200.0,144.0,3.2,20.0,120.0,13.0,0.7,473.0,34.0,213.119995
4,1,83.20917,1.0,76.4,26.060587,18,5,SICU,1,Tuesday,...,249.0,137.0,4.8,19.0,104.0,29.0,2.0,251.0,35.0,NaN


In [74]:
#subtasks

subtask1 = df_train.day_28_flg.isna().sum()

df_1 = df_train[df_train.day_28_flg.isna() == False]
subtask2_1 = len(df_1[(df_1.sofa_first > 10) & (df_1.day_28_flg == 1)]) / len(df_1[df_1.sofa_first > 10]) * 100
subtask2_2 = len(df_1[(df_1.sofa_first <= 10) & (df_1.day_28_flg == 1)]) / len(df_1[df_1.sofa_first <= 10]) * 100

df_subtasks = pd.DataFrame({
    "subtaskID": [1, 2, 2],
    "datapointID": [1, 1, 2],
    "answer": [subtask1, subtask2_1, round(subtask2_2, 2)]
})

df_subtasks

,subtaskID,datapointID,answer
0,1,1,1235.00
1,2,1,0.00
2,2,2,51.02


In [75]:
features = [col for col in df_test.columns if col != 'ID']
target = 'day_28_flg'

X = df_1[features]
y = df_1[target]

df_2 = df_train[df_train.day_28_flg.isna() == True]
X2 = df_2[features]

X_final = df_test[features]

cat_cols = ["day_icu_intime", "service_unit"]
from sklearn.preprocessing import LabelEncoder
for col in cat_cols:
    enc = LabelEncoder()
    X.loc[:, col] = enc.fit_transform(X.loc[:, col])
    X2.loc[:, col] = enc.transform(X2.loc[:, col])
    X_final.loc[:, col] = enc.transform(X_final.loc[:, col])

from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=features)
X2 = pd.DataFrame(imputer.transform(X2), columns=features)
X_final = pd.DataFrame(imputer.transform(X_final), columns=features)

X.head()

,aline_flg,age,gender_num,weight_first,bmi,sapsi_first,sofa_first,service_unit,service_num,day_icu_intime,...,platelet_first,sodium_first,potassium_first,tco2_first,chloride_first,bun_first,creatinine_first,po2_first,pco2_first,iv_day_1
0,1.0,72.36841,1.0,75.00,29.912791,15.0,9.0,2.0,1.0,0.0,...,354.0,138.0,4.6,15.0,109.0,41.0,1.6,196.0,39.0,2230.875000
1,0.0,48.40044,0.0,73.25,24.161440,14.0,5.0,1.0,0.0,1.0,...,367.0,133.0,3.7,19.0,98.0,25.0,0.9,473.0,32.0,2850.000000
2,1.0,48.97547,1.0,97.00,31.579563,13.0,11.0,2.0,1.0,4.0,...,145.0,133.0,4.0,31.0,95.0,18.0,1.0,493.0,34.0,3682.880127
3,1.0,60.37829,0.0,51.00,25.207321,18.0,7.0,2.0,1.0,2.0,...,200.0,144.0,3.2,20.0,120.0,13.0,0.7,473.0,34.0,213.119995
4,1.0,83.20917,1.0,76.40,26.060587,18.0,5.0,2.0,1.0,5.0,...,249.0,137.0,4.8,19.0,104.0,29.0,2.0,251.0,35.0,1134.629944


In [76]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

model = RandomForestClassifier()

cross_val_score(model, X, y, cv=5, scoring='roc_auc').mean()

np.float64(1.0)

In [77]:
model.fit(X, y)
probs = model.predict_proba(X2)[:, 1]

In [78]:
mask = (probs > 0.9) | (probs < 0.1)

X2_conf = X2[mask]
y2_conf = (probs[mask] > 0.5).astype(int)

X = pd.concat([X, X2_conf])
y = pd.concat([y, pd.Series(y2_conf)])

In [79]:
from sklearn.linear_model import LogisticRegression

model2 = LogisticRegression(max_iter=1000000)

cross_val_score(model2, X, y, cv=5, scoring='roc_auc')

C:\Users\Ilie-Goga\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of f AND g EVALUATIONS EXCEEDS LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\Ilie-Goga\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of f AND g EVALUATIONS EXCEEDS LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.or

array([0.77659574, 1.        , 1.        , 0.99148936, 1.        ])

In [80]:
model2.fit(X, y)
y_preds = model2.predict_proba(X_final)[:, 1]

C:\Users\Ilie-Goga\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of f AND g EVALUATIONS EXCEEDS LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [81]:
df_preds = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.ID,
    "answer": y_preds
})

df_preds.head()

,subtaskID,datapointID,answer
0,3,0,0.048620
1,3,1,0.918404
2,3,2,0.158547
3,3,3,0.007370
4,3,4,0.011094


In [82]:
submission = pd.concat([df_subtasks, df_preds])
submission

,subtaskID,datapointID,answer
0,1,1,1235.000000
1,2,1,0.000000
2,2,2,51.020000
0,3,0,0.048620
1,3,1,0.918404
...,...,...,...
345,3,345,0.574394
346,3,346,0.466025
347,3,347,0.096524
348,3,348,0.995292


In [83]:
submission.to_csv("submission.csv", index=False)